# Retention Playbook

Goal: turn the segments (`Customer_Segmentation.ipynb`) and churn risk
scores (`Churn_Prediction_Model.ipynb`) into **specific actions**, not
vague advice.

The problem statement is explicit about this: "'Reduce discounts' is not
an acceptable answer" (from the retail problem, but the same standard
applies here) - every recommendation needs to name **who** receives it,
**what** the offer is, **when** it triggers, and **why**.

Picks up from `customer_segments.csv`.

In [1]:
import pandas as pd

df = pd.read_csv('customer_segments.csv')
print(df.shape)
df[['Loyalty Number', 'segment_name', 'churn_risk_tier', 'churn_risk_score', 'CLV']].head(3)

(13699, 36)


,Loyalty Number,segment_name,churn_risk_tier,churn_risk_score,CLV
0,480934,Occasional Travelers (moderate activity),Low Risk,0.229263,3839.14
1,549612,"Champions (high value, high engagement)",Low Risk,0.278039,3839.61
2,608370,"Champions (high value, high engagement)",Low Risk,0.201028,3839.75


## 1. Defining the playbook

For every (segment, risk tier) combination, we assign one specific action.
The logic is written as a function rather than a lookup table so the
reasoning is visible and easy to defend - not just a set of arbitrary
rules.

In [2]:
def recommend_action(segment_name, risk_tier):
    if segment_name == 'Champions (high value, high engagement)' and risk_tier in ['Medium Risk', 'High Risk']:
        return (
            'Priority save call',
            'Personal outreach from a loyalty concierge + a status match/upgrade offer for their next 2 trips',
            'Within 7 days of being flagged',
            'These are the highest-value, most-engaged members - losing one costs far more than a personal call.',
        )
    if segment_name == 'At-Risk (used to fly, going quiet)':
        return (
            'Win-back bonus points',
            'Automated email offering 2,000 bonus points on their next booking, valid for 60 days',
            'Immediately once Months Since Last Flight crosses 6',
            'They have a proven flying habit - a time-limited nudge is usually enough to bring them back before the habit fully breaks.',
        )
    if segment_name == 'Occasional Travelers (moderate activity)' and risk_tier in ['Medium Risk', 'High Risk']:
        return (
            'Route-relevant offer',
            'Targeted discount on routes they have flown before, timed to the season they historically travel in',
            'Seasonal, timed to their own past travel pattern',
            'They already have a preferred route/season - relevance beats a generic blanket discount.',
        )
    if segment_name == 'Dormant / Never Flew':
        return (
            'Low-cost reactivation nudge',
            'One low-cost email campaign per quarter with a simple "welcome back" offer; no calls, no heavy discounts',
            'Quarterly, automated only',
            'Lowest historical value and no flying pattern to build on - heavy investment here is not worth it, but staying present costs little.',
        )
    if segment_name == 'Loyal Flyers (steady, dependable)' and risk_tier in ['Medium Risk', 'High Risk']:
        return (
            'Loyalty check-in call',
            'A personal check-in call (not a discount) asking what changed, paired with a small tenure-based perk (e.g. free seat upgrade)',
            'Within 14 days of being flagged',
            'A steady flyer showing rising risk is a signal something specific changed (job, life event, a bad experience) - '
            'a discount will not fix that, but ignoring it risks losing a dependable, proven customer.',
        )
    if segment_name == 'Loyal Flyers (steady, dependable)':
        return (
            'Recognition, not discounting',
            'A "thank you" status update / early access to seat selection, no discount needed',
            'Ongoing, low-frequency',
            'They already fly reliably and show no rising risk - discounting here just gives away margin for behavior they were going to do anyway.',
        )
    return ('Monitor', 'No action needed this cycle', 'N/A', 'Behavior currently healthy.')

## 2. Building the playbook table

One row per (segment, risk tier) combination that actually occurs in the
data, with the size and average value of that group attached - so this
reads as a business table, not just a set of rules.

In [3]:
playbook_rows = []
for segment_name in df['segment_name'].unique():
    for risk_tier in df['churn_risk_tier'].unique():
        subset = df[(df['segment_name'] == segment_name) & (df['churn_risk_tier'] == risk_tier)]
        if len(subset) == 0:
            continue
        action, offer, timing, reason = recommend_action(segment_name, risk_tier)
        playbook_rows.append({
            'Segment': segment_name,
            'Risk Tier': risk_tier,
            'Number of Members': len(subset),
            'Avg CLV': round(subset['CLV'].mean(), 0),
            'Recommended Action': action,
            'Offer Details': offer,
            'Trigger Timing': timing,
            'Why': reason,
        })

playbook = pd.DataFrame(playbook_rows).sort_values(by=['Risk Tier', 'Avg CLV'], ascending=[True, False])
playbook

,Segment,Risk Tier,Number of Members,Avg CLV,Recommended Action,Offer Details,Trigger Timing,Why
8,"Loyal Flyers (steady, dependable)",High Risk,41,16055.0,Loyalty check-in call,A personal check-in call (not a discount) aski...,Within 14 days of being flagged,A steady flyer showing rising risk is a signal...
10,Dormant / Never Flew,High Risk,430,8515.0,Low-cost reactivation nudge,One low-cost email campaign per quarter with a...,"Quarterly, automated only",Lowest historical value and no flying pattern ...
2,Occasional Travelers (moderate activity),High Risk,207,7329.0,Route-relevant offer,Targeted discount on routes they have flown be...,"Seasonal, timed to their own past travel pattern",They already have a preferred route/season - r...
5,"Champions (high value, high engagement)",High Risk,45,7253.0,Priority save call,Personal outreach from a loyalty concierge + a...,Within 7 days of being flagged,"These are the highest-value, most-engaged memb..."
6,"Loyal Flyers (steady, dependable)",Low Risk,2194,14371.0,"Recognition, not discounting","A ""thank you"" status update / early access to ...","Ongoing, low-frequency",They already fly reliably and show no rising r...
3,"Champions (high value, high engagement)",Low Risk,6277,6364.0,Monitor,No action needed this cycle,N/A,Behavior currently healthy.
0,Occasional Travelers (moderate activity),Low Risk,2444,6143.0,Monitor,No action needed this cycle,N/A,Behavior currently healthy.
7,"Loyal Flyers (steady, dependable)",Medium Risk,452,12636.0,Loyalty check-in call,A personal check-in call (not a discount) aski...,Within 14 days of being flagged,A steady flyer showing rising risk is a signal...
9,Dormant / Never Flew,Medium Risk,196,7391.0,Low-cost reactivation nudge,One low-cost email campaign per quarter with a...,"Quarterly, automated only",Lowest historical value and no flying pattern ...
1,Occasional Travelers (moderate activity),Medium Risk,1202,6715.0,Route-relevant offer,Targeted discount on routes they have flown be...,"Seasonal, timed to their own past travel pattern",They already have a preferred route/season - r...


## 3. Per-member action list

What the ops team would actually load into their outreach tool - one row
per member with their specific recommended action attached.

In [4]:
df['recommended_action'] = df.apply(lambda row: recommend_action(row['segment_name'], row['churn_risk_tier'])[0], axis=1)

priority_members = df[df['churn_risk_tier'] == 'High Risk'].sort_values('CLV', ascending=False)
print(f"{len(priority_members)} members are High Risk. Top 10 by CLV (action these first):")
priority_members[['Loyalty Number', 'segment_name', 'CLV', 'churn_risk_score', 'recommended_action']].head(10)

723 members are High Risk. Top 10 by CLV (action these first):


,Loyalty Number,segment_name,CLV,churn_risk_score,recommended_action
13314,615459,"Loyal Flyers (steady, dependable)",83325.38,0.662333,Loyalty check-in call
3410,838263,"Loyal Flyers (steady, dependable)",67907.27,0.974877,Loyalty check-in call
4764,333051,Dormant / Never Flew,51426.25,0.894375,Low-cost reactivation nudge
8004,737027,Dormant / Never Flew,49221.43,0.975033,Low-cost reactivation nudge
3396,258296,"Loyal Flyers (steady, dependable)",44856.11,0.832724,Loyalty check-in call
3388,266561,Dormant / Never Flew,42462.97,0.674937,Low-cost reactivation nudge
3375,767971,Dormant / Never Flew,40636.67,0.991573,Low-cost reactivation nudge
7986,148810,Dormant / Never Flew,40224.01,0.987394,Low-cost reactivation nudge
13676,851979,Dormant / Never Flew,39033.08,0.991258,Low-cost reactivation nudge
3370,856496,Dormant / Never Flew,38496.95,0.614087,Low-cost reactivation nudge


## 4. Estimated revenue impact

A simple, defensible business-impact number: CLV weighted by each
member's churn probability, summed across everyone. This is what turns
"we built a model" into "here's what it's worth to act on it."


In [5]:
revenue_at_risk = (df['CLV'] * df['churn_risk_score']).sum()
high_risk_clv = priority_members['CLV'].sum()

print(f'Estimated revenue at risk (CLV x churn probability, summed): ${revenue_at_risk:,.0f}')
print(f'Total CLV sitting in the High Risk tier alone: ${high_risk_clv:,.0f}')

Estimated revenue at risk (CLV x churn probability, summed): $30,088,400
Total CLV sitting in the High Risk tier alone: $6,163,354


## 5. Save

`retention_playbook.csv` is the summary table for the strategy memo.
`customer_action_list.csv` is the operational file - every member, their
risk, their segment, and their specific recommended action. This is also
what feeds the Streamlit dashboard.

In [6]:
playbook.to_csv('retention_playbook.csv', index=False)
df.to_csv('customer_action_list.csv', index=False)

print('Saved retention_playbook.csv and customer_action_list.csv')

Saved retention_playbook.csv and customer_action_list.csv
